# 03 Попытка в рельное решение

Задача предсказать куда пойдет пара на следующий день

In [2]:
import importlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import models, timeseries_cv
importlib.reload(models); importlib.reload(timeseries_cv)
from models import (GRUWithAttention, WindowDataset, SoftDirectionalHuberLoss,
                    CURATED_FEATURES, z_feature_cols, get_device)
from timeseries_cv import (cross_validate, summarize_cv, compare_strategies,
                           evaluate_holdout)
from models import save_checkpoint, load_checkpoint
from pathlib import Path

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = get_device()
CKPT_PATH = Path('checkpoints/gru_attention_btc_eth.pt')
print('device:', DEVICE)

device: mps


## Загрузка данных, ограничение до BTC + ETH

Объединяем train+val в пул для CV; test полностью откладываем для финальной
holdout-оценки.

In [3]:
HEADLINE = ['BTC-USD', 'ETH-USD']

def load(split):
    return pd.read_csv(f'data/{split}_features.csv', index_col=0, parse_dates=True)

train_df = load('train'); val_df = load('val'); test_df = load('test')

def restrict(df, coins):
    return df[df['ticker'].isin(coins)].sort_index()

cv_df_h   = restrict(pd.concat([train_df, val_df]), HEADLINE)
test_df_h = restrict(test_df, HEADLINE)

print('CV pool :', len(cv_df_h), 'rows |', cv_df_h.index.min().date(), '->', cv_df_h.index.max().date())
print('Holdout :', len(test_df_h), 'rows |', sorted(test_df_h["ticker"].unique()))

CV pool : 4368 rows | 2019-07-18 -> 2025-07-09
Holdout : 488 rows | ['BTC-USD', 'ETH-USD']


In [4]:
# Feature set: the curated core-8 (high-signal, low-variance choice).
feature_cols = list(CURATED_FEATURES)
# Sanity: all present in the data.
missing = [c for c in feature_cols if c not in cv_df_h.columns]
assert not missing, f'missing features: {missing}'
print(f'features ({len(feature_cols)}):', feature_cols)
print(f'(full z_* set available: {len(z_feature_cols(cv_df_h))} cols — swap in for an ablation)')

features (8): ['bull_regime', 'z_bb_pct_b', 'z_volume_z20', 'z_log_close_return_1', 'z_range', 'z_ret_autocorr', 'dow_cos', 'high_vol_regime']
(full z_* set available: 43 cols — swap in for an ablation)


## Time-series кросс-валидация: expanding vs sliding

Обе стратегии нарезают фолды по **дате через все тикеры**; затем `WindowDataset`
строит последовательности внутри каждого тикера. Явно передаём
`SoftDirectionalHuberLoss` (стандарт проекта) и используем расписание
warmup→cosine.

In [5]:
SEQ_LEN = 20
criterion = SoftDirectionalHuberLoss(delta=1.01, dir_weight=0.8, sharpness=10.0)

# Single source of truth for the architecture — also saved into the checkpoint.
ARCH_CONFIG = dict(input_size=len(feature_cols), hidden_size=32,
                   num_layers=3, linear_hidden=32, dropout=0.1)

def model_factory(seed=SEED):
    torch.manual_seed(seed)
    return GRUWithAttention(**ARCH_CONFIG).to(DEVICE)

shared = dict(
    full_df=cv_df_h, feature_cols=feature_cols,
    model_factory=model_factory, dataset_cls=WindowDataset,
    n_folds=5, val_frac=0.10, seq_len=SEQ_LEN, batch_size=128,
    lr=1e-3, epochs=1000, patience=13, huber_delta=1.01,
    device=DEVICE, criterion=criterion, weight_decay=1e-1, warmup_epochs=10,
)

In [6]:
exp_results = cross_validate(**shared, strategy='expanding')
exp_df = summarize_cv(exp_results)
exp_df


  [EXPANDING] Fold 0  |  train 2019-07-18 → 2021-12-06  |  val 2021-12-07 → 2022-07-12
  Epoch   1  train_loss=0.7406  val_loss=0.7332  dir_acc=0.533
  Epoch   2  train_loss=0.5748  val_loss=0.4567  dir_acc=0.533
  Epoch   3  train_loss=0.3954  val_loss=0.4817  dir_acc=0.472
  Epoch   4  train_loss=0.3793  val_loss=0.4215  dir_acc=0.545
  Epoch   5  train_loss=0.3769  val_loss=0.4052  dir_acc=0.535
  Epoch   6  train_loss=0.3902  val_loss=0.3931  dir_acc=0.528
  Epoch   7  train_loss=0.3659  val_loss=0.4451  dir_acc=0.462
  Epoch   8  train_loss=0.3947  val_loss=0.4215  dir_acc=0.530
  Epoch   9  train_loss=0.3892  val_loss=0.4354  dir_acc=0.492
  Epoch  10  train_loss=0.3606  val_loss=0.3978  dir_acc=0.548
  Epoch  11  train_loss=0.4104  val_loss=0.4282  dir_acc=0.505
  Epoch  12  train_loss=0.3760  val_loss=0.4044  dir_acc=0.530
  Epoch  13  train_loss=0.3727  val_loss=0.3848  dir_acc=0.545
  Epoch  14  train_loss=0.3642  val_loss=0.4091  dir_acc=0.535
  Epoch  15  train_loss=0.3748

,fold_idx,strategy,train_loss,val_loss,val_mae,val_dir_acc,n_train,n_val,train_start,train_end,val_start,val_end,best_model_state,val_index
0,0,expanding,0.324581,0.313989,0.777524,0.570707,1706,396,2019-07-18,2021-12-06,2021-12-07,2022-07-12,"{'gru.weight_ih_l0': [[tensor(0.1418, device='...","DatetimeIndex(['2021-12-27', '2021-12-28', '20..."
1,1,expanding,0.381268,0.264292,0.626312,0.542929,2252,396,2019-07-18,2022-09-05,2022-09-06,2023-04-11,"{'gru.weight_ih_l0': [[tensor(0.1421, device='...","DatetimeIndex(['2022-09-26', '2022-09-27', '20..."
2,2,expanding,0.386413,0.341897,0.698884,0.547980,2798,396,2019-07-18,2023-06-05,2023-06-06,2024-01-09,"{'gru.weight_ih_l0': [[tensor(0.1390, device='...","DatetimeIndex(['2023-06-26', '2023-06-27', '20..."
3,3,expanding,0.372766,0.292900,0.685969,0.542929,3344,396,2019-07-18,2024-03-04,2024-03-05,2024-10-08,"{'gru.weight_ih_l0': [[tensor(0.1445, device='...","DatetimeIndex(['2024-03-25', '2024-03-26', '20..."
4,4,expanding,0.372768,0.279728,0.679316,0.558081,3892,396,2019-07-18,2024-12-03,2024-12-04,2025-07-09,"{'gru.weight_ih_l0': [[tensor(0.1439, device='...","DatetimeIndex(['2024-12-24', '2024-12-25', '20..."


In [7]:
sli_results = cross_validate(**shared, strategy='sliding')
sli_df = summarize_cv(sli_results)
sli_df


  [SLIDING] Fold 0  |  train 2019-07-18 → 2021-12-06  |  val 2021-12-07 → 2022-07-12
  Epoch   1  train_loss=0.7406  val_loss=0.7332  dir_acc=0.533
  Epoch   2  train_loss=0.5748  val_loss=0.4567  dir_acc=0.533
  Epoch   3  train_loss=0.3954  val_loss=0.4817  dir_acc=0.472
  Epoch   4  train_loss=0.3793  val_loss=0.4215  dir_acc=0.545
  Epoch   5  train_loss=0.3769  val_loss=0.4052  dir_acc=0.535
  Epoch   6  train_loss=0.3902  val_loss=0.3931  dir_acc=0.528
  Epoch   7  train_loss=0.3659  val_loss=0.4451  dir_acc=0.462
  Epoch   8  train_loss=0.3947  val_loss=0.4215  dir_acc=0.530
  Epoch   9  train_loss=0.3892  val_loss=0.4354  dir_acc=0.492
  Epoch  10  train_loss=0.3606  val_loss=0.3978  dir_acc=0.548
  Epoch  11  train_loss=0.4104  val_loss=0.4282  dir_acc=0.505
  Epoch  12  train_loss=0.3760  val_loss=0.4044  dir_acc=0.530
  Epoch  13  train_loss=0.3727  val_loss=0.3848  dir_acc=0.545
  Epoch  14  train_loss=0.3642  val_loss=0.4091  dir_acc=0.535
  Epoch  15  train_loss=0.3748  

,fold_idx,strategy,train_loss,val_loss,val_mae,val_dir_acc,n_train,n_val,train_start,train_end,val_start,val_end,best_model_state,val_index
0,0,sliding,0.324581,0.313989,0.777524,0.570707,1706,396,2019-07-18,2021-12-06,2021-12-07,2022-07-12,"{'gru.weight_ih_l0': [[tensor(0.1418, device='...","DatetimeIndex(['2021-12-27', '2021-12-28', '20..."
1,1,sliding,0.363287,0.240671,0.618123,0.583333,1706,396,2020-04-16,2022-09-05,2022-09-06,2023-04-11,"{'gru.weight_ih_l0': [[tensor(0.1395, device='...","DatetimeIndex(['2022-09-26', '2022-09-27', '20..."
2,2,sliding,0.338172,0.340384,0.700144,0.550505,1706,396,2021-01-14,2023-06-05,2023-06-06,2024-01-09,"{'gru.weight_ih_l0': [[tensor(0.1402, device='...","DatetimeIndex(['2023-06-26', '2023-06-27', '20..."
3,3,sliding,0.347631,0.286969,0.686049,0.537879,1706,396,2021-10-14,2024-03-04,2024-03-05,2024-10-08,"{'gru.weight_ih_l0': [[tensor(0.1510, device='...","DatetimeIndex(['2024-03-25', '2024-03-26', '20..."
4,4,sliding,0.339416,0.300933,0.679056,0.570707,1706,396,2022-07-15,2024-12-03,2024-12-04,2025-07-09,"{'gru.weight_ih_l0': [[tensor(0.1490, device='...","DatetimeIndex(['2024-12-24', '2024-12-25', '20..."


In [8]:
comp_df = compare_strategies(exp_results, sli_results)
comp_df


  STRATEGY COMPARISON
  expanding   |  dir_acc = 0.553 ± 0.011  |  val_loss = 0.2986 ± 0.0271
  sliding     |  dir_acc = 0.563 ± 0.016  |  val_loss = 0.2966 ± 0.0330

  → Sliding wins: older data may be hurting. Consider shorter history.
  → Expanding is more stable across folds (lower variance).


,strategy,val_loss_mean,val_loss_std,dir_acc_mean,dir_acc_std,n_folds,best_fold_acc,worst_fold_acc
0,expanding,0.298561,0.027119,0.552525,0.010642,5,0.570707,0.542929
1,sliding,0.296589,0.033042,0.562626,0.016240,5,0.583333,0.537879


## Holdout-оценка (test) против бейзлайнов, по тикерам + агрегат

In [9]:
holdout = evaluate_holdout(
    train_df=cv_df_h, eval_df=test_df_h,
    feature_cols=feature_cols, model_factory=model_factory,
    dataset_cls=WindowDataset, lr=1e-3, device=DEVICE,
    seq_len=SEQ_LEN, batch_size=128, epochs=1000, patience=13,
    huber_delta=1.01, criterion=criterion, weight_decay=1e-1, warmup_epochs=10,
    ticker=HEADLINE,
)
print(f'Aggregate holdout directional accuracy: {holdout.eval_dir_acc:.4f}')
print(f'Holdout loss: {holdout.eval_loss:.4f}')
print('\nPer-ticker:')
for tk, m in holdout.per_ticker.items():
    print(f'  {tk:8s}: dir_acc {m["dir_acc"]:.4f}')


  HOLDOUT EVALUATION  |  ticker=BTC-USD, ETH-USD
  train 2019-07-18 -> 2025-07-09  (4,368 rows)
  eval  2025-07-10 -> 2026-03-10  (488 rows)
  Epoch   1  train_loss=0.7297  eval_loss=0.7687  dir_acc=0.475
  Epoch   2  train_loss=0.4429  eval_loss=0.4483  dir_acc=0.513
  Epoch   3  train_loss=0.3554  eval_loss=0.4105  dir_acc=0.540
  Epoch   4  train_loss=0.3705  eval_loss=0.3984  dir_acc=0.551
  Epoch   5  train_loss=0.3746  eval_loss=0.4101  dir_acc=0.554
  Epoch   6  train_loss=0.3465  eval_loss=0.4280  dir_acc=0.536
  Epoch   7  train_loss=0.3720  eval_loss=0.3787  dir_acc=0.558
  Epoch   8  train_loss=0.3483  eval_loss=0.3860  dir_acc=0.556
  Epoch   9  train_loss=0.3554  eval_loss=0.4004  dir_acc=0.558
  Epoch  10  train_loss=0.3573  eval_loss=0.4360  dir_acc=0.536
  Epoch  11  train_loss=0.3604  eval_loss=0.4210  dir_acc=0.547
  Epoch  12  train_loss=0.3373  eval_loss=0.4185  dir_acc=0.542
  Epoch  13  train_loss=0.3308  eval_loss=0.3950  dir_acc=0.545
  Epoch  14  train_loss=0.

## Экспорт чекпойнта

In [10]:
save_checkpoint(
    holdout.model, CKPT_PATH,
    config=ARCH_CONFIG, feature_cols=feature_cols, seq_len=SEQ_LEN,
    arch='GRUWithAttention',
    metrics={'holdout_dir_acc': float(holdout.eval_dir_acc),
             'holdout_loss': float(holdout.eval_loss),
             'tickers': HEADLINE},
)
print('saved checkpoint ->', CKPT_PATH)

# Sanity: reload and confirm holdout dir-acc is reproduced from the saved weights.
reloaded, payload = load_checkpoint(CKPT_PATH, device=DEVICE)
print('reloaded arch:', payload['arch'], '| features:', len(payload['feature_cols']),
      '| saved metrics:', payload['metrics'])

saved checkpoint -> checkpoints/gru_attention_btc_eth.pt
reloaded arch: GRUWithAttention | features: 8 | saved metrics: {'holdout_dir_acc': 0.5580357313156128, 'holdout_loss': 0.3787465989589691, 'tickers': ['BTC-USD', 'ETH-USD']}


In [11]:
# Naive baselines on the same holdout for context.
from models import directional_accuracy
y = test_df_h['target'].values
maj = np.sign(np.nanmean(y))  # always-up / always-down
print(f'Majority-sign baseline : {directional_accuracy(np.full_like(y, maj), y):.4f}')
print(f'Always-up baseline     : {directional_accuracy(np.ones_like(y), y):.4f}')
print(f'PCA+Logistic baseline  : 0.5593  (from experiments/pca_svm_baseline.ipynb)')
print(f'GRU+Attention (ours)   : {holdout.eval_dir_acc:.4f}')

Majority-sign baseline : 0.5287
Always-up baseline     : 0.4713
PCA+Logistic baseline  : 0.5593  (from experiments/pca_svm_baseline.ipynb)
GRU+Attention (ours)   : 0.5580


## Сравнение архитектур

GRU+Attention — основная (прогнана вживую выше). Остальные строки — **значения
последнего прогона** holdout из ноутбуков-источников в `experiments/`
(перезапустите, чтобы обновить). Они используют общие `models.py` /
`timeseries_cv.py`, поэтому сравнение «яблоки к яблокам».

In [12]:
comparison = pd.DataFrame([
    {'model': 'GRU + Attenti (headline, live)', 'cv_dir_acc_expanding': 0.552,
     'holdout_dir_acc': round(float(holdout.eval_dir_acc), 4), 'source': 'this notebook'},
    {'model': 'GRU + Attention (advanced)', 'cv_dir_acc_expanding': 0.541,
     'holdout_dir_acc': 0.534, 'source': 'experiments/gru_attention_advanced.ipynb'},
    {'model': 'GRU + Attention (iter2)', 'cv_dir_acc_expanding': 0.549,
     'holdout_dir_acc': 0.509, 'source': 'experiments/gru_attention_iter2.ipynb'},
    {'model': 'Two-head GRU (dir + magnitude)', 'cv_dir_acc_expanding': None,
     'holdout_dir_acc': 0.543, 'source': 'experiments/two_head_gru_attention.ipynb'},
    {'model': 'Dir + Vol combined (dir-only)', 'cv_dir_acc_expanding': None,
     'holdout_dir_acc': 0.568, 'source': 'experiments/dir_vol_combined.ipynb'},
    {'model': 'PCA + Logistic (classical baseline)', 'cv_dir_acc_expanding': None,
     'holdout_dir_acc': 0.541, 'source': 'experiments/pca_svm_baseline.ipynb'},
])
comparison

,model,cv_dir_acc_expanding,holdout_dir_acc,source
0,"GRU + Attenti (headline, live)",0.552,0.558,this notebook
1,GRU + Attention (advanced),0.541,0.534,experiments/gru_attention_advanced.ipynb
2,GRU + Attention (iter2),0.549,0.509,experiments/gru_attention_iter2.ipynb
3,Two-head GRU (dir + magnitude),NaN,0.543,experiments/two_head_gru_attention.ipynb
4,Dir + Vol combined (dir-only),NaN,0.568,experiments/dir_vol_combined.ipynb
5,PCA + Logistic (classical baseline),NaN,0.541,experiments/pca_svm_baseline.ipynb


## Выводы

- С новой задачей модель справляется лучше
- Скользящее окно примерно одниково по метрикам с расширающимся (variance на расширщемся ниже)
- 55-56% - Это потолок для OCHLV метрик (судя по схожим моделям)